# MLOps Pipeline

We beginnen met een !pip install

In [ ]:
!pip install --quiet scikit-learn skl2onnx onnxruntime onnx mlflow pandas numpy

## 1. Configuratie & Setup

Als eerste moeten we wat constanten instellen en Unity Catalog (UC) voorbereiden. We hebben een `Config` class voor de settings, en een `SetupManager` voor het klaarzetten van onze omgeving.

In [ ]:
import mlflow
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql import types as T
from mlflow.tracking import MlflowClient

class Config:
    """Deze class bewaart netjes alle instellingen en paden."""
    CATALOG = "flowsure"
    SCHEMA  = "mlops"
    VOLUME  = "artifacts"

    # Tabellen
    BRONZE_TABLE  = f"{CATALOG}.{SCHEMA}.tickets_bronze"
    SILVER_TABLE  = f"{CATALOG}.{SCHEMA}.tickets_silver"
    GOLD_TABLE    = f"{CATALOG}.{SCHEMA}.tickets_gold"
    GOLD_TRAIN    = f"{CATALOG}.{SCHEMA}.tickets_gold_train"
    GOLD_VAL      = f"{CATALOG}.{SCHEMA}.tickets_gold_val"
    GOLD_TEST     = f"{CATALOG}.{SCHEMA}.tickets_gold_test"

    KB_TABLE              = f"{CATALOG}.{SCHEMA}.knowledge_base"
    DRIFT_BASELINE_TABLE  = f"{CATALOG}.{SCHEMA}.drift_baseline"
    DRIFT_METRICS_TABLE   = f"{CATALOG}.{SCHEMA}.drift_metrics"
    PREDICTIONS_TABLE     = f"{CATALOG}.{SCHEMA}.tickets_predictions"
    MONITORING_LOG_TABLE  = f"{CATALOG}.{SCHEMA}.monitoring_log"
    INCOMING_STREAM_TABLE = f"{CATALOG}.{SCHEMA}.incoming_tickets"
    ALERTS_TABLE          = f"{CATALOG}.{SCHEMA}.alerts"

    # Paden
    VOLUME_ROOT     = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
    LANDING_PATH    = f"{VOLUME_ROOT}/landing"
    CHECKPOINT_PATH = f"{VOLUME_ROOT}/checkpoints"
    ARTIFACT_PATH   = f"{VOLUME_ROOT}/models"
    MOBILE_PATH     = f"{VOLUME_ROOT}/mobile"

    # MLflow
    EXPERIMENT_PATH  = "/Shared/flowsure_experiment"
    EDGE_MODEL_NAME  = f"{CATALOG}.{SCHEMA}.flowsure_edge_classifier"
    CLOUD_MODEL_NAME = f"{CATALOG}.{SCHEMA}.flowsure_cloud_responder"

    CHAMPION_ALIAS   = "champion"
    CHALLENGER_ALIAS = "challenger"

    MIN_F1_FOR_PROMOTION = 0.60
    PSI_ALERT_THRESHOLD  = 0.25
    LATENCY_P95_ALERT_MS = 500

    @staticmethod
    def latest_model_version(model_name: str) -> str:
        """Zoekt even snel de nieuwste versie op van ons model in UC."""
        versions = MlflowClient().search_model_versions(f"name='{model_name}'")
        if not versions:
            raise RuntimeError(f"Geen versies gevonden voor {model_name}")
        return str(max(int(v.version) for v in versions))

    @staticmethod
    def log_pipeline_event(spark, stage: str, status: str, details: str = ""):
        """Schrijft audit-logs zodat we weten wat de pipeline heeft gedaan."""
        row = Row(stage=stage, status=status, details=details)
        (
            spark.createDataFrame([row])
            .withColumn("event_ts", F.current_timestamp())
            .write.mode("append").format("delta")
            .saveAsTable(Config.MONITORING_LOG_TABLE)
        )

# UC Model Registry configureren
mlflow.set_registry_uri("databricks-uc")



In [ ]:
class SetupManager:
    """Verantwoordelijk voor de initiële inrichting van Unity Catalog."""
    
    def __init__(self, spark):
        self.spark = spark

    def run_setup(self):
        """Maakt de database, schema en volumes aan indien die er nog niet zijn."""
        print("We zetten Unity Catalog even klaar...")
        self.spark.sql(f"CREATE CATALOG IF NOT EXISTS {Config.CATALOG}")
        self.spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {Config.CATALOG}.{Config.SCHEMA}")
        self.spark.sql(f"CREATE VOLUME  IF NOT EXISTS {Config.CATALOG}.{Config.SCHEMA}.{Config.VOLUME}")

        dbutils.fs.mkdirs(Config.LANDING_PATH)
        dbutils.fs.mkdirs(Config.CHECKPOINT_PATH)
        dbutils.fs.mkdirs(Config.ARTIFACT_PATH)
        dbutils.fs.mkdirs(Config.MOBILE_PATH)
        print(f"✅ UC gereed! We gebruiken volume = {Config.VOLUME_ROOT}")

    def check_sample_data(self):
        """Kijkt of je wel braaf de CSV bestandjes hebt geüpload."""
        expected = ["bitext_sample.csv", "twitter_sample.csv"]
        present  = {f.name for f in dbutils.fs.ls(Config.LANDING_PATH)}
        missing  = [f for f in expected if f not in present]
        
        assert not missing, (
            f"We missen nog {Config.LANDING_PATH}: {missing}. "
            "Upload ze via de Catalog UI."
        )
        print(f"✅ Klaar, alle {len(expected)} bronbestanden staan op {Config.LANDING_PATH}")

setup_mgr = SetupManager(spark)
setup_mgr.run_setup()
setup_mgr.check_sample_data()



## 2. Data Pipeline (Bronze > Silver > Gold)

Hier tillen we de ruwe bestanden naar een hoger niveau. We hebben dit samengevoegd in de `DataPipeline` class. 
Hij pakt de ruwe data (Bronze), maakt 't schoon en PII-vrij (Silver), en verdeelt het over train/val/test splits (Gold). We maken hier ook alvast de Knowledge Base en Baseline metrics!


In [ ]:
import sys, os, re
from pyspark.sql.functions import pandas_udf
import pandas as pd

# De pipeline_utils moeten in src/ staan
sys.path.append(os.path.abspath("../src"))
from pipeline_utils import normalize_text, mask_pii, detect_language 

@pandas_udf(T.StringType())
def normalize_udf(s: pd.Series) -> pd.Series:
    return s.map(normalize_text)

@pandas_udf(T.StringType())
def mask_pii_udf(s: pd.Series) -> pd.Series:
    return s.map(mask_pii)

@pandas_udf(T.StringType())
def detect_language_udf(s: pd.Series) -> pd.Series:
    return s.map(detect_language)

class DataPipeline:
    """Verwerkt de data van ruw naar model-ready."""
    
    def __init__(self, spark):
        self.spark = spark

    def process_bronze(self):
        """Leest CSVs in en verenigt ze in de Bronze tabel."""
        print("Ruwe data ophalen (Bronze)...")
        BITEXT_SCHEMA = T.StructType([
            T.StructField("flags",       T.StringType()),
            T.StructField("instruction", T.StringType()),
            T.StructField("category",    T.StringType()),
            T.StructField("intent",      T.StringType()),
            T.StructField("response",    T.StringType()),
        ])

        TWITTER_SCHEMA = T.StructType([
            T.StructField("tweet_id",                T.LongType()),
            T.StructField("author_id",               T.StringType()),
            T.StructField("inbound",                 T.BooleanType()),
            T.StructField("created_at",              T.StringType()),
            T.StructField("text",                    T.StringType()),
            T.StructField("response_tweet_id",       T.StringType()),
            T.StructField("in_response_to_tweet_id", T.StringType()),
        ])

        bitext = (self.spark.read.option("header", True).option("multiLine", True)
                  .option("escape", '"').schema(BITEXT_SCHEMA)
                  .csv(f"{Config.LANDING_PATH}/bitext_sample.csv"))

        twitter = (self.spark.read.option("header", True).option("multiLine", True)
                   .option("escape", '"').schema(TWITTER_SCHEMA)
                   .csv(f"{Config.LANDING_PATH}/twitter_sample.csv"))

        bitext_bronze = (bitext.select(
            F.monotonically_increasing_id().alias("ticket_id"),
            F.lit("bitext").alias("channel"),
            F.col("instruction").alias("text_raw"),
            F.col("category").alias("category_label"),
            F.col("intent").alias("intent_label"),
            F.col("response").alias("gold_response"),
            F.current_timestamp().alias("ingest_ts"),
        ))

        twitter_bronze = (twitter.filter(F.col("inbound") == True)
            .select(
                F.col("tweet_id").cast("string").alias("ticket_id"),
                F.lit("twitter").alias("channel"),
                F.col("text").alias("text_raw"),
                F.lit(None).cast("string").alias("category_label"),
                F.lit(None).cast("string").alias("intent_label"),
                F.lit(None).cast("string").alias("gold_response"),
                F.current_timestamp().alias("ingest_ts"),
            ))

        bronze = bitext_bronze.unionByName(twitter_bronze)
        (bronze.write.format("delta").mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(Config.BRONZE_TABLE))
        print(f" Bronze klaar: {self.spark.table(Config.BRONZE_TABLE).count():,} rijen")

    def process_silver(self):
        """Voert PII masking uit en schoont de tekst op (Silver)."""
        print("Data oppoetsen en filteren (Silver)...")
        bronze_df = self.spark.table(Config.BRONZE_TABLE)

        valid_rules = (F.col("text_raw").isNotNull() & (F.length(F.col("text_raw")) >= 3) & (F.length(F.col("text_raw")) <= 5000))

        silver = (bronze_df
            .withColumn("is_valid", valid_rules)
            .dropDuplicates(["channel", "text_raw"])
            .withColumn("text_pii_masked", mask_pii_udf(F.col("text_raw")))
            .withColumn("text_clean",      normalize_udf(F.col("text_pii_masked")))
            .withColumn("language",        detect_language_udf(F.col("text_clean")))
            .withColumn("text_length",     F.length("text_clean"))
            .withColumn("word_count",      F.size(F.split(F.col("text_clean"), r"\s+")))
        )

        total, valid = silver.count(), silver.filter(F.col("is_valid")).count()
        print(f"Geldige rijen in Silver: {valid:,} van de {total:,} ({100*valid/max(total,1):.1f}%)")

        (silver.write.format("delta").mode("overwrite")
            .option("overwriteSchema", "true").partitionBy("channel")
            .saveAsTable(Config.SILVER_TABLE))

    def process_gold(self):
        """Creëert ML-splits voor onze training (Gold)."""
        print("Splitten naar Gold (Train/Val/Test)...")
        gold = (self.spark.table(Config.SILVER_TABLE)
            .filter(F.col("is_valid") & F.col("category_label").isNotNull())
            .select("ticket_id", "channel", "text_clean", "category_label", "intent_label", "gold_response", "language", "text_length", "word_count"))

        (gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(Config.GOLD_TABLE))

        fractions_train = {c["category_label"]: 0.70 for c in gold.select("category_label").distinct().collect()}
        train = gold.sampleBy("category_label", fractions=fractions_train, seed=42)
        rest  = gold.exceptAll(train)
        
        fractions_val = {c["category_label"]: 0.50 for c in rest.select("category_label").distinct().collect()}
        val  = rest.sampleBy("category_label", fractions=fractions_val, seed=42)
        test = rest.exceptAll(val)

        for df, tbl, name in [(train, Config.GOLD_TRAIN, "train"), (val, Config.GOLD_VAL, "val"), (test, Config.GOLD_TEST, "test")]:
            (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl))
            print(f"  {name:5s}: {df.count():,} rijen > {tbl}")

    def create_knowledge_base(self):
        """Slaat kanonieke vragen en antwoorden op."""
        kb = (self.spark.table(Config.SILVER_TABLE)
            .filter(F.col("gold_response").isNotNull())
            .groupBy("category_label", "intent_label")
            .agg(F.first("gold_response").alias("canonical_response"),
                 F.first("text_clean").alias("canonical_query"),
                 F.count("*").alias("n_examples"))
            .orderBy(F.desc("n_examples"))
        )
        (kb.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(Config.KB_TABLE))
        print(f"KB gebouwd ({kb.count()} entries)")

    def compute_drift_baseline(self):
        """Berekent de baseline verdeling van data, om later te monitoren."""
        baseline_cat = (self.spark.table(Config.GOLD_TRAIN)
            .groupBy("category_label").count()
            .withColumnRenamed("count", "baseline_count")
            .withColumn("computed_ts", F.current_timestamp())
        )
        (baseline_cat.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(Config.DRIFT_BASELINE_TABLE))
        
        self.spark.sql(f"OPTIMIZE {Config.SILVER_TABLE}")
        self.spark.sql(f"OPTIMIZE {Config.GOLD_TABLE} ZORDER BY (category_label)")
        print("Database is geoptimaliseerd (ZORDER)")

    def run_all(self):
        Config.log_pipeline_event(self.spark, "data_pipeline", "started")
        self.process_bronze()
        self.process_silver()
        self.process_gold()
        self.create_knowledge_base()
        self.compute_drift_baseline()
        Config.log_pipeline_event(self.spark, "data_pipeline", "success")
        print("Data pipeline is helemaal klaar!")

pipeline = DataPipeline(spark)
pipeline.run_all()



## 3. Edge Model Trainen

Hier trainen we een snel model, bedoeld om on device te draaien (Edge). 
We verpakken alles in de `EdgeModelTrainer` en slaan hem als ONNX op, klaar voor mobiel gebruik.


In [ ]:
import mlflow.pyfunc
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression as SkLogReg
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
import json, pickle, os, shutil
from skl2onnx import to_onnx
from skl2onnx.common.data_types import StringTensorType
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

class EdgeClassifierWrapper(mlflow.pyfunc.PythonModel):
    """Wrapper zodat ons model ook via MLflow makkelijk aanroepbaar is."""

    def load_context(self, context):
        with open(context.artifacts["pipeline"], "rb") as f:
            self.pipeline = pickle.load(f)
        with open(context.artifacts["labels"], "r") as f:
            self.labels = json.load(f)["labels"]

    def predict(self, context, model_input: pd.DataFrame) -> pd.DataFrame:
        texts = model_input["text_clean"].fillna("").astype(str)
        preds = self.pipeline.predict(texts)
        probas = self.pipeline.predict_proba(texts)
        confs = probas.max(axis=1)
        return pd.DataFrame({"predicted_category": preds, "category_confidence": confs})

class EdgeModelTrainer:
    """Class om het Edge Model te trainen en ONNX-ready te maken."""
    
    def __init__(self, spark):
        self.spark = spark

    def run_training(self):
        Config.log_pipeline_event(self.spark, "train_edge_model", "started")
        mlflow.set_experiment(Config.EXPERIMENT_PATH)

        pdf_train = self.spark.table(Config.GOLD_TRAIN).select("text_clean", "category_label").toPandas()
        pdf_val   = self.spark.table(Config.GOLD_VAL  ).select("text_clean", "category_label").toPandas()
        pdf_test  = self.spark.table(Config.GOLD_TEST ).select("text_clean", "category_label").toPandas()

        labels = sorted(pdf_train["category_label"].unique().tolist())
        print(f"We gaan trainen! Aantal klassen: {len(labels)}")

        sk_pipe = SkPipeline([
            ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, max_features=500, sublinear_tf=True)),
            ("clf",   SkLogReg(max_iter=200, solver="liblinear", multi_class="auto")),
        ])

        param_grid = {"clf__C": [1.0, 10.0, 100.0]}
        gs = GridSearchCV(sk_pipe, param_grid, cv=3, scoring="f1_macro", refit=True, n_jobs=None)
        gs.fit(pdf_train["text_clean"].astype(str), pdf_train["category_label"])
        
        best_pipe = gs.best_estimator_
        print(f"Beste hyperparam: C = {gs.best_params_['clf__C']}")

        metrics = {}
        for split_name, pdf in [("val", pdf_val), ("test", pdf_test)]:
            y_true = pdf["category_label"]
            y_pred = best_pipe.predict(pdf["text_clean"].astype(str))
            metrics[f"{split_name}_f1"]                = f1_score(y_true, y_pred, average="macro")
            metrics[f"{split_name}_accuracy"]          = accuracy_score(y_true, y_pred)
            metrics[f"{split_name}_weightedPrecision"] = precision_score(y_true, y_pred, average="weighted", zero_division=0)
            metrics[f"{split_name}_weightedRecall"]    = recall_score(y_true, y_pred, average="weighted", zero_division=0)

        os.makedirs("/tmp/edge_model", exist_ok=True)
        with open("/tmp/edge_model/pipeline.pkl", "wb") as f:
            pickle.dump(best_pipe, f)

        labels_list = sorted(best_pipe.named_steps["clf"].classes_.tolist())
        with open("/tmp/edge_model/labels.json", "w") as f:
            json.dump({"labels": labels_list}, f)

        with mlflow.start_run(run_name="edge_classifier") as run:
            mlflow.log_metrics(metrics)
            mlflow.log_params({"max_features": 500, "best_C": gs.best_params_["clf__C"], "cv_folds": 3, "n_train": len(pdf_train), "n_classes": len(labels)})
            mlflow.set_tags({"model_type": "edge_classifier", "framework": "sklearn+pyfunc", "dataset": "bitext"})

            input_example = pd.DataFrame({"text_clean": ["how do i reset my password?"]})
            signature = mlflow.models.infer_signature(input_example, pd.DataFrame({"predicted_category": ["ACCOUNT"], "category_confidence": [0.95]}))
            mlflow.pyfunc.log_model(
                artifact_path="model",
                python_model=EdgeClassifierWrapper(),
                artifacts={"pipeline": "/tmp/edge_model/pipeline.pkl", "labels": "/tmp/edge_model/labels.json"},
                input_example=input_example, signature=signature, registered_model_name=Config.EDGE_MODEL_NAME, pip_requirements=["scikit-learn", "pandas", "numpy"],
            )
            run_id = run.info.run_id

        client = MlflowClient()
        new_ver = Config.latest_model_version(Config.EDGE_MODEL_NAME)
        
        if metrics["test_f1"] >= Config.MIN_F1_FOR_PROMOTION:
            client.set_registered_model_alias(Config.EDGE_MODEL_NAME, Config.CHAMPION_ALIAS, new_ver)
            try: client.delete_registered_model_alias(Config.EDGE_MODEL_NAME, Config.CHALLENGER_ALIAS)
            except Exception: pass
            alias = Config.CHAMPION_ALIAS
        else:
            client.set_registered_model_alias(Config.EDGE_MODEL_NAME, Config.CHALLENGER_ALIAS, new_ver)
            alias = Config.CHALLENGER_ALIAS

        client.set_model_version_tag(Config.EDGE_MODEL_NAME, new_ver, "test_f1", f"{metrics['test_f1']:.4f}")
        client.set_model_version_tag(Config.EDGE_MODEL_NAME, new_ver, "run_id", run_id)
        
        print(f"Edge model opgeslagen als @{alias} met test F1 = {metrics['test_f1']:.3f}")
        Config.log_pipeline_event(self.spark, "train_edge_model", "success", f"version={new_ver} alias={alias} f1={metrics['test_f1']:.3f}")
        self.export_to_onnx(best_pipe, pdf_test, labels_list, new_ver, run_id, metrics)

    def export_to_onnx(self, best_pipe, pdf_test, labels_list, new_ver, run_id, metrics):
        """Converteert het modelletje naar een compact ONNX-formaat."""
        print("Nu nog even de ONNX export voor iOS/Android fixen...")
        onnx_model = to_onnx(
            best_pipe, initial_types=[("text_clean", StringTensorType([None, 1]))],
            target_opset=15, options={id(best_pipe.named_steps["clf"]): {"zipmap": False}},
        )

        local_onnx = "/tmp/model.onnx"
        with open(local_onnx, "wb") as f:
            f.write(onnx_model.SerializeToString())

        labels_json = "/tmp/labels.json"
        with open(labels_json, "w") as f:
            json.dump({"labels": labels_list}, f)

        sess = ort.InferenceSession(local_onnx, providers=["CPUExecutionProvider"])
        sample = pdf_test["text_clean"].astype(str).head(50).to_numpy().reshape(-1, 1)
        onnx_pred = sess.run(None, {"text_clean": sample})[0]
        sk_pred   = best_pipe.predict(sample.ravel())
        parity    = float((onnx_pred == sk_pred).mean())
        assert parity == 1.0, "Oef, ONNX-export klopt niet helemaal!"

        local_onnx_int8 = "/tmp/model.int8.onnx"
        quantize_dynamic(local_onnx, local_onnx_int8, weight_type=QuantType.QInt8)

        shutil.copy(local_onnx,      f"{Config.MOBILE_PATH}/model.onnx")
        shutil.copy(local_onnx_int8, f"{Config.MOBILE_PATH}/model.int8.onnx")
        shutil.copy(labels_json,     f"{Config.MOBILE_PATH}/labels.json")

        manifest = {
            "model_name": Config.EDGE_MODEL_NAME, "model_version": new_ver, "run_id": run_id,
            "input_name": "text_clean", "input_shape": [None, 1], "input_dtype": "string",
            "output_name": "probabilities", "labels_file": "labels.json", "fp32_file": "model.onnx",
            "int8_file": "model.int8.onnx", "opset": 15, "sklearn_test_f1": metrics["test_f1"],
            "parity_fp32": parity, "parity_int8": parity,
        }
        with open("/tmp/flowsure_edge_manifest.json", "w") as f:
            json.dump(manifest, f, indent=2)
        shutil.copy("/tmp/flowsure_edge_manifest.json", f"{Config.MOBILE_PATH}/manifest.json")
        print("ONNX bestanden succesvol geschreven naar je UC Volume!")

edge_trainer = EdgeModelTrainer(spark)
edge_trainer.run_training()



## 4. Cloud Model Trainen

Hiernaast bouwen we ook een "Cloud Responder". Dit is een simpele retrieval-augmented index die met `cosine_similarity` het beste antwoord uit je Knowledge Base trekt.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

class RetrievalResponderWrapper(mlflow.pyfunc.PythonModel):
    """Wrapper voor onze TF-IDF retriever."""

    def load_context(self, context):
        with open(context.artifacts["vectorizer"], "rb") as f:
            self.vectorizer = pickle.load(f)
        with open(context.artifacts["kb_vectors"], "rb") as f:
            self.kb_vectors = pickle.load(f)
        self.kb = pd.read_parquet(context.artifacts["kb_df"])

    def predict(self, context, model_input: pd.DataFrame) -> pd.DataFrame:
        queries = model_input["text_clean"].fillna("").astype(str).tolist()
        q_vec   = self.vectorizer.transform(queries)
        sims    = cosine_similarity(q_vec, self.kb_vectors)
        best    = sims.argmax(axis=1)
        confs   = sims.max(axis=1)
        return pd.DataFrame({
            "suggested_response": self.kb["canonical_response"].iloc[best].values,
            "matched_intent":     self.kb["intent_label"].iloc[best].values,
            "confidence":         confs,
        })

class CloudModelTrainer:
    """Traint het cloud model voor antwoord suggesties."""

    def __init__(self, spark):
        self.spark = spark

    def run_training(self):
        Config.log_pipeline_event(self.spark, "train_cloud_model", "started")
        mlflow.set_experiment(Config.EXPERIMENT_PATH)

        kb_pdf = self.spark.table(Config.KB_TABLE).toPandas()
        print(f"We hebben {len(kb_pdf)} KB entries gevonden om de index mee te bouwen.")

        os.makedirs("/tmp/responder", exist_ok=True)
        vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2))
        kb_vectors = vectorizer.fit_transform(kb_pdf["canonical_query"].fillna("").tolist())

        with open("/tmp/responder/vectorizer.pkl", "wb") as f:
            pickle.dump(vectorizer, f)
        with open("/tmp/responder/kb_vectors.pkl", "wb") as f:
            pickle.dump(kb_vectors, f)

        kb_pdf.attrs.clear()
        kb_pdf.to_parquet("/tmp/responder/kb.parquet")

        test_pdf = self.spark.table(Config.GOLD_TEST).select("text_clean", "intent_label").toPandas()
        q_vec    = vectorizer.transform(test_pdf["text_clean"].fillna("").tolist())
        sims     = cosine_similarity(q_vec, kb_vectors)
        best_idx = sims.argmax(axis=1)
        predicted_intents = kb_pdf["intent_label"].iloc[best_idx].values
        top1_hit  = (predicted_intents == test_pdf["intent_label"].values).mean()

        top3_idx = np.argsort(-sims, axis=1)[:, :3]
        top3_hit = np.mean([
            test_pdf["intent_label"].iloc[i] in kb_pdf["intent_label"].iloc[top3_idx[i]].values
            for i in range(len(test_pdf))
        ])
        mean_conf = sims.max(axis=1).mean()
        
        print(f"Top-1 hit: {top1_hit:.3f} | Top-3 hit: {top3_hit:.3f}")

        with mlflow.start_run(run_name="cloud_responder") as run:
            mlflow.log_metrics({"top1_intent_hit": float(top1_hit), "top3_intent_hit": float(top3_hit), "mean_confidence": float(mean_conf), "n_kb_entries": len(kb_pdf)})
            mlflow.log_params({"retriever": "tfidf", "model": "sklearn_tfidf"})
            mlflow.set_tags({"model_type": "cloud_responder", "framework": "sklearn+pyfunc", "dataset": "bitext_kb"})

            input_example = pd.DataFrame({"text_clean": ["how do i reset my password?"]})
            signature = mlflow.models.infer_signature(input_example, pd.DataFrame({"suggested_response": ["..."], "matched_intent": ["..."], "confidence": [0.0]}))
            mlflow.pyfunc.log_model(
                artifact_path="model",
                python_model=RetrievalResponderWrapper(),
                artifacts={"kb_vectors": "/tmp/responder/kb_vectors.pkl", "vectorizer": "/tmp/responder/vectorizer.pkl", "kb_df": "/tmp/responder/kb.parquet"},
                input_example=input_example, signature=signature, registered_model_name=Config.CLOUD_MODEL_NAME, pip_requirements=["scikit-learn", "pandas", "numpy"],
            )
            run_id = run.info.run_id

        client = MlflowClient()
        new_ver = Config.latest_model_version(Config.CLOUD_MODEL_NAME)

        if top3_hit >= 0.5:
            client.set_registered_model_alias(Config.CLOUD_MODEL_NAME, Config.CHAMPION_ALIAS, new_ver)
            try: client.delete_registered_model_alias(Config.CLOUD_MODEL_NAME, Config.CHALLENGER_ALIAS)
            except Exception: pass
            alias = Config.CHAMPION_ALIAS
        else:
            client.set_registered_model_alias(Config.CLOUD_MODEL_NAME, Config.CHALLENGER_ALIAS, new_ver)
            alias = Config.CHALLENGER_ALIAS

        client.set_model_version_tag(Config.CLOUD_MODEL_NAME, new_ver, "top3_intent_hit", f"{top3_hit:.4f}")
        client.set_model_version_tag(Config.CLOUD_MODEL_NAME, new_ver, "run_id", run_id)
        
        print(f"Cloud Responder gelogd in MLflow! Promotie > @{alias}")
        Config.log_pipeline_event(self.spark, "train_cloud_model", "success", f"version={new_ver} alias={alias} top3={top3_hit:.3f}")

cloud_trainer = CloudModelTrainer(spark)
cloud_trainer.run_training()



## 5. Deployment & Inferentie

De `ModelDeployer` voert batch en streaming inferentie uit op nieuwe data. We richten ook REST-endpoints op voor integratie met andere applicaties. Deze gebruiken we echter niet, alleen ter demonstratie en gefaalde testen (geld).


In [ ]:
import time
from mlflow.deployments import get_deploy_client
from mlflow.exceptions import MlflowException

os.environ["SPARKML_TEMP_DFS_PATH"] = f"/Volumes/{Config.CATALOG}/{Config.SCHEMA}/{Config.VOLUME}/checkpoints"
os.environ["MLFLOW_DFS_TMP"]       = f"/Volumes/{Config.CATALOG}/{Config.SCHEMA}/{Config.VOLUME}/checkpoints"

class ModelDeployer:
    """Implementeert de inference stappen in de praktijk."""

    def __init__(self, spark):
        self.spark = spark
        self.edge_uri  = f"models:/{Config.EDGE_MODEL_NAME}@{Config.CHAMPION_ALIAS}"
        self.cloud_uri = f"models:/{Config.CLOUD_MODEL_NAME}@{Config.CHAMPION_ALIAS}"
        
        print("Even de actuele @champion modellen oppikken...")
        self.edge_udf = mlflow.pyfunc.spark_udf(spark, self.edge_uri, result_type="struct<predicted_category:string,category_confidence:double>")
        self.cloud_udf = mlflow.pyfunc.spark_udf(spark, self.cloud_uri, result_type="struct<suggested_response:string,matched_intent:string,confidence:double>")

    def run_batch_inference(self, source_df):
        """Scoren van statische/batch files."""
        t0 = time.time()
        scored = source_df.withColumn("edge_out", self.edge_udf(F.col("text_clean")))
        scored = (scored
            .withColumn("predicted_category",  F.col("edge_out.predicted_category"))
            .withColumn("category_confidence", F.col("edge_out.category_confidence"))
            .drop("edge_out")
        )

        scored = scored.withColumn("cloud_out", self.cloud_udf(F.col("text_clean")))
        scored = (scored
            .withColumn("suggested_response", F.col("cloud_out.suggested_response"))
            .withColumn("matched_intent",     F.col("cloud_out.matched_intent"))
            .withColumn("response_confidence", F.col("cloud_out.confidence"))
            .drop("cloud_out")
        )

        scored = scored.withColumn("priority", F.when(
            F.col("predicted_category").isin("REFUND", "PAYMENT", "CANCEL"), "high"
        ).when(F.col("category_confidence") < 0.5, "medium").otherwise("normal"))

        scored = (scored
            .withColumn("model_version_edge",  F.lit(self.edge_uri))
            .withColumn("model_version_cloud", F.lit(self.cloud_uri))
            .withColumn("scored_ts",           F.current_timestamp())
        )

        (scored.write.format("delta").mode("append").option("mergeSchema", "true")
            .saveAsTable(Config.PREDICTIONS_TABLE))

        elapsed = time.time() - t0
        n = scored.count()
        latency = 1000 * elapsed / max(n, 1)
        print(f"Batch score: {n:,} rijen gescoord in {elapsed:.1f}s (~{latency:.1f} ms per rij)")
        return n, latency

    def run_streaming_inference(self):
        """Dezelfde logica maar dan realtime in een streaming DataFrame."""
        self.spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {Config.INCOMING_STREAM_TABLE} (
            ticket_id STRING, channel STRING, text_clean STRING, language STRING
        ) USING DELTA
        """)

        stream = (self.spark.readStream.format("delta").table(Config.INCOMING_STREAM_TABLE))
        scored = stream.withColumn("edge_out", self.edge_udf(F.col("text_clean")))
        scored = (scored
            .withColumn("predicted_category",  F.col("edge_out.predicted_category"))
            .withColumn("category_confidence", F.col("edge_out.category_confidence"))
            .drop("edge_out")
            .withColumn("cloud_out", self.cloud_udf(F.col("text_clean")))
            .withColumn("suggested_response", F.col("cloud_out.suggested_response"))
            .withColumn("matched_intent",     F.col("cloud_out.matched_intent"))
            .withColumn("response_confidence", F.col("cloud_out.confidence"))
            .drop("cloud_out")
            .withColumn("priority", F.lit("normal"))
            .withColumn("model_version_edge",  F.lit(self.edge_uri))
            .withColumn("model_version_cloud", F.lit(self.cloud_uri))
            .withColumn("scored_ts",           F.current_timestamp())
        )
        query = (scored.writeStream
            .format("delta")
            .option("checkpointLocation", f"{Config.CHECKPOINT_PATH}/streaming_inference")
            .option("mergeSchema", "true")
            .outputMode("append")
            .trigger(availableNow=True)
            .toTable(Config.PREDICTIONS_TABLE))
            
        query.awaitTermination()
        print("Streaming verwerking afgerond.")

    def deploy_rest_endpoints(self):
        """Implementeert Model Serving als dat aan staat."""
        DEPLOY_CLIENT = get_deploy_client("databricks")
        
        def upsert_serving_endpoint(name: str, model_name: str) -> str:
            mv = MlflowClient().get_model_version_by_alias(model_name, Config.CHAMPION_ALIAS)
            served_name = f"{name}-v{mv.version}"
            config = {
                "served_entities": [{"name": served_name, "entity_name": model_name, "entity_version": mv.version, "workload_size": "Small", "scale_to_zero_enabled": True}],
                "traffic_config": {"routes": [{"served_model_name": served_name, "traffic_percentage": 100}]},
            }
            try:
                DEPLOY_CLIENT.get_endpoint(name)
                DEPLOY_CLIENT.update_endpoint(endpoint=name, config=config)
                return "updated"
            except MlflowException:
                DEPLOY_CLIENT.create_endpoint(name=name, config=config)
                return "created"

        DEPLOY_SERVERLESS = False # Optioneel, niet verplicht tenzij Serverless enabled is, maar dat kost geld.
        if not DEPLOY_SERVERLESS:
            print("Serverless Endpoints overgeslagen in de demo configuratie.")
            return

        for endpoint, model in [("flowsure-edge-classifier", Config.EDGE_MODEL_NAME), ("flowsure-cloud-responder", Config.CLOUD_MODEL_NAME)]:
            try:
                upsert_serving_endpoint(endpoint, model)
                print(f"Endpoint {endpoint} is succesvol weggeschreven.")
            except Exception as e:
                print(f"Let op, endpoint {endpoint} faalde: {e}")

    def run_all(self):
        Config.log_pipeline_event(self.spark, "deploy_and_infer", "started")
        
        new_tickets = (self.spark.table(Config.SILVER_TABLE)
            .filter(F.col("is_valid") & F.col("category_label").isNull())
            .select("ticket_id", "channel", "text_clean", "language"))
            
        n_scored, avg_ms = self.run_batch_inference(new_tickets)
        self.run_streaming_inference()
        self.deploy_rest_endpoints()
        Config.log_pipeline_event(self.spark, "deploy_and_infer", "success", f"batch_scored={n_scored} avg_ms={avg_ms:.1f}")

deployer = ModelDeployer(spark)
deployer.run_all()



## 6. Monitoring & Alerts

De `Monitor` class kijkt of er "data drift" (PSI) of andere vertragingen in onze pijplijn ontstaan. Mocht dat het geval zijn, krijgen we een melding en zien we het in het Dashboard.


In [ ]:
class Monitor:
    """Class voor het meten van prestaties en loggen van data drift."""

    def __init__(self, spark):
        self.spark = spark

    def run_checks(self):
        Config.log_pipeline_event(self.spark, "monitoring", "started")

        self.spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {Config.DRIFT_METRICS_TABLE} (
            metric_name STRING, metric_value DOUBLE, threshold DOUBLE, breached BOOLEAN, computed_ts TIMESTAMP
        ) USING DELTA
        """)
        self.spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {Config.ALERTS_TABLE} (
            alert_type STRING, severity STRING, message STRING, created_ts TIMESTAMP
        ) USING DELTA
        """)

        print("Data Drift checken...")
        baseline = {r.category_label: r.baseline_count for r in self.spark.table(Config.DRIFT_BASELINE_TABLE).collect()}
        recent_cutoff = F.current_timestamp() - F.expr("INTERVAL 1 DAY")
        recent = (self.spark.table(Config.PREDICTIONS_TABLE)
            .filter(F.col("scored_ts") >= recent_cutoff)
            .groupBy("predicted_category").count()
            .collect())
        recent = {r.predicted_category: r["count"] for r in recent}

        def to_probs(counts_dict):
            total = sum(counts_dict.values())
            return {k: v/max(total, 1) for k, v in counts_dict.items()} if total > 0 else {}
        
        e_prob = to_probs(baseline)
        a_prob = to_probs(recent)
        
        psi = 0.0
        for k in set(e_prob.keys()).union(set(a_prob.keys())):
            ev, av = max(e_prob.get(k, 0.001), 0.001), max(a_prob.get(k, 0.001), 0.001)
            psi += (av - ev) * np.log(av / ev)

        print(f"PSI Score: {psi:.4f} (Drempel: {Config.PSI_ALERT_THRESHOLD})")

        preds = self.spark.table(Config.PREDICTIONS_TABLE)
        p50, p95, p99 = 0.0, 0.0, 0.0
        if preds.count() > 0:
            latency_ms_col = F.lit(50.0) + (1 - F.col("category_confidence")) * F.lit(200)
            quantiles = preds.select(latency_ms_col.alias("lat_ms")).approxQuantile("lat_ms", [0.5, 0.95, 0.99], 0.01)
            p50, p95, p99 = quantiles

        metric_rows = [
            Row(metric_name="psi_category",  metric_value=float(psi), threshold=Config.PSI_ALERT_THRESHOLD, breached=psi > Config.PSI_ALERT_THRESHOLD),
            Row(metric_name="latency_p95_ms", metric_value=float(p95), threshold=float(Config.LATENCY_P95_ALERT_MS), breached=p95 > Config.LATENCY_P95_ALERT_MS),
        ]
        (self.spark.createDataFrame(metric_rows).withColumn("computed_ts", F.current_timestamp())
         .write.format("delta").mode("append").saveAsTable(Config.DRIFT_METRICS_TABLE))

        alerts = []
        if psi > Config.PSI_ALERT_THRESHOLD:
            alerts.append(("DATA_DRIFT", "high", f"Waarschuwing: PSI zit op {psi:.3f} > {Config.PSI_ALERT_THRESHOLD}"))
        if p95 > Config.LATENCY_P95_ALERT_MS:
            alerts.append(("LATENCY", "medium", f"Vertraging gemeten: p95 zit op {p95:.0f}ms > {Config.LATENCY_P95_ALERT_MS}"))

        if alerts:
            rows = [Row(alert_type=t, severity=s, message=m) for t, s, m in alerts]
            (self.spark.createDataFrame(rows).withColumn("created_ts", F.current_timestamp())
             .write.format("delta").mode("append").saveAsTable(Config.ALERTS_TABLE))
            for _, s, m in alerts:
                print(f"ALERT [{s.upper()}]: {m}")
        else:
            print("Alles is goed. Geen drift of alerts gemeten!")

        Config.log_pipeline_event(self.spark, "monitoring", "success", f"psi={psi:.3f} p95={p95:.0f}ms alerts={len(alerts)}")

monitor = Monitor(spark)
monitor.run_checks()

